In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost", "openpyxl", "joblib"])

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

TARGET = "case_type"
TEST_SIZE = 0.20
RANDOM_STATE = 42
WINDOWS_PATH = r"F:\dataset\supreme_court_judgments_enhanced.xlsx"

if os.path.exists(WINDOWS_PATH):
    FILE_PATH = WINDOWS_PATH
else:
    try:
        from google.colab import files
        print("Please upload supreme_court_judgments_enhanced.xlsx")
        uploaded = files.upload()
        excel_files = [name for name in uploaded.keys() if name.lower().endswith((".xlsx", ".xls"))]
        if not excel_files:
            raise FileNotFoundError("No Excel file uploaded.")
        FILE_PATH = excel_files[0]
    except ImportError:
        excel_files = [f for f in os.listdir(".") if f.lower().endswith((".xlsx", ".xls"))]
        if not excel_files:
            raise FileNotFoundError("Excel dataset not found.")
        FILE_PATH = excel_files[0]

print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

print("Dataset:", FILE_PATH)

df = pd.read_excel(FILE_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.columns = df.columns.astype(str).str.strip()

if TARGET not in df.columns:
    matches = [c for c in df.columns if c.lower() == TARGET.lower()]
    if matches:
        TARGET = matches[0]
    else:
        raise ValueError(f"Target column '{TARGET}' not found.")

df = df.dropna(subset=[TARGET]).copy()

df[TARGET] = df[TARGET].astype(str).str.strip()

df = df[df[TARGET] != ""].copy()

df = df.drop_duplicates().reset_index(drop=True)

print("\nTarget:", TARGET)

print("\nClass distribution:")
print(df[TARGET].value_counts())

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

remove_columns = []

for col in X.columns:
    name = str(col).lower()
    unique_count = X[col].nunique(dropna=True)
    unique_ratio = unique_count / len(X)

    if (
        ("id" in name or
         "case_no" in name or
         "case_number" in name or
         "url" in name or
         "link" in name or
         "filename" in name)
        and unique_ratio > 0.70
    ):
        remove_columns.append(col)

if remove_columns:
    print("\nRemoving identifier columns:")
    for col in remove_columns:
        print(" -", col)
    X = X.drop(columns=remove_columns)

text_columns = []
categorical_columns = []
numeric_columns = []

for col in X.columns:

    if pd.api.types.is_numeric_dtype(X[col]):
        numeric_columns.append(col)

    elif X[col].dtype == "object":

        sample = X[col].dropna().astype(str).head(1000)

        if len(sample) > 0:
            avg_length = sample.str.len().mean()
            unique_count = X[col].nunique(dropna=True)

            if avg_length > 40 or unique_count > 100:
                text_columns.append(col)
            else:
                categorical_columns.append(col)
        else:
            categorical_columns.append(col)

print("\nText columns detected:")
for col in text_columns:
    print(" -", col)

print("\nCategorical columns:")
for col in categorical_columns:
    print(" -", col)

print("\nNumeric columns:")
for col in numeric_columns:
    print(" -", col)

X = X.drop(columns=text_columns, errors="ignore")

for col in numeric_columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[col] = X[col].fillna(X[col].median())

for col in categorical_columns:
    X[col] = X[col].fillna("Unknown").astype(str)

X = X.replace([np.inf, -np.inf], np.nan)

for col in X.columns:
    if pd.api.types.is_numeric_dtype(X[col]):
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna("Unknown").astype(str)

categorical_columns = [
    col for col in X.columns
    if X[col].dtype == "object"
]

feature_columns = X.columns.tolist()

cat_features = [
    X.columns.get_loc(col)
    for col in categorical_columns
]

print("\nTotal features:", len(feature_columns))
print("Categorical features:", len(cat_features))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\n" + "=" * 70)
print("TRAINING FAST CATBOOST")
print("=" * 70)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.08,
    depth=7,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=RANDOM_STATE,
    l2_leaf_reg=5,
    random_strength=0.5,
    border_count=64,
    bootstrap_type="Bernoulli",
    subsample=0.8,
    verbose=50,
    early_stopping_rounds=50,
    allow_writing_files=False,
    thread_count=-1
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

print("\nTraining completed.")

print("\n" + "=" * 70)
print("PREDICTION")
print("=" * 70)

y_pred = model.predict(X_test)

y_pred = np.asarray(y_pred).reshape(-1)

accuracy = accuracy_score(y_test, y_pred)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"Accuracy       : {accuracy * 100:.2f}%")
print(f"Weighted F1    : {weighted_f1 * 100:.2f}%")
print(f"Macro F1       : {macro_f1 * 100:.2f}%")

print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=0
    )
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

labels = sorted(y.unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print(cm_df)

results = pd.DataFrame({
    "Actual_case_type": y_test.values,
    "Predicted_case_type": y_pred
})

results["Correct"] = (
    results["Actual_case_type"]
    ==
    results["Predicted_case_type"]
)

results.to_excel(
    "case_type_catboost_predictions.xlsx",
    index=False
)

report = classification_report(
    y_test,
    y_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()

report_df.to_excel(
    "case_type_catboost_classification_report.xlsx"
)

model.save_model(
    "case_type_catboost_model.cbm"
)

joblib.dump(
    feature_columns,
    "case_type_catboost_features.pkl"
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("case_type_catboost_predictions.xlsx")
print("case_type_catboost_classification_report.xlsx")
print("case_type_catboost_model.cbm")
print("case_type_catboost_features.pkl")

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)

print(f"\nFinal Accuracy: {accuracy * 100:.2f}%")
print(f"Weighted F1: {weighted_f1 * 100:.2f}%")
print(f"Macro F1: {macro_f1 * 100:.2f}%")

Please upload supreme_court_judgments_enhanced.xlsx


Saving supreme_court_judgments_enhanced.xlsx to supreme_court_judgments_enhanced.xlsx
LOADING DATASET
Dataset: supreme_court_judgments_enhanced.xlsx
Rows: 26688
Columns: 68

Target: case_type

Class distribution:
case_type
civil             9822
criminal          5785
constitutional    4569
tax               4097
service           1305
election           458
family             410
corporate          225
uncategorized       17
Name: count, dtype: int64

Removing identifier columns:
 - filename

Text columns detected:
 - case_number
 - citation
 - judgment_date
 - judges
 - bench
 - headnotes
 - acts_referred
 - petitioner
 - respondent
 - sections_applied
 - penalties
 - imprisonment
 - fines
 - appellant

Categorical columns:
 - case_subtype
 - charges
 - appeal_level
 - citation_pattern
 - judges_mentioned
 - primary_judge
 - primary_expertise_area
 - assignment_priority

Numeric columns:
 - year
 - page_count
 - word_count
 - character_count
 - sentence_count
 - paragraph_count
 - av